In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [17]:
df=pd.read_csv('Student_Performance.csv')
df.head(3)

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0


In [18]:
x=df.drop(['Performance Index'],axis=1)
y=df['Performance Index']
x.head(3)

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced
0,7,99,Yes,9,1
1,4,82,No,4,2
2,8,51,Yes,7,2


In [19]:
x['Extracurricular Activities']=x['Extracurricular Activities'].replace('Yes',1).replace('No',0).astype(int)

In [20]:
x.head(3)

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced
0,7,99,1,9,1
1,4,82,0,4,2
2,8,51,1,7,2


In [22]:
sc=StandardScaler()
x=sc.fit_transform(x)
x=pd.DataFrame(x)
x.head(3)

,0,1,2,3,4
0,0.775188,1.704176,1.010455,1.456205,-1.249754
1,-0.383481,0.723913,-0.989654,-1.492294,-0.900982
2,1.161410,-1.063626,1.010455,0.276805,-0.900982


In [24]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8)

In [27]:
reg1=LinearRegression(n_jobs=-1)
reg1.fit(xtrain,ytrain)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",-1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[ 7.4 ,17.68, 0.29, 0.81, 0.55]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,55.2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,5
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](5,)","[90.91,89.98,89.65,88.43,88.04]"


In [28]:
from sklearn.metrics import r2_score
ac1=r2_score(ytest,reg1.predict(xtest))
ac2=r2_score(ytrain,reg1.predict(xtrain))
print(f"Test score {ac1} , Train score {ac2}")


Test score 0.9877303604760199 , Train score 0.9889838909234047


In [29]:
from sklearn.linear_model import SGDRegressor 

In [49]:
sg=SGDRegressor(early_stopping=True,average=True)
pm1={
    'loss' : ['squared_error','huber'],
    'penalty' : ['l1','l2'],
    'alpha' : [0.0001,0.001,0.01,0.1],
    'learning_rate' : ['constant','adaptive'],
    'eta0' : [0.01,0.05,0.1,0.5],
    'max_iter' :[10000]

}
pm2={
    'loss' : ['squared_error','huber'],
    'penalty' : ['elasticnet'],
    'l1_ratio' : [0.1,0.3,0.5,0.7],
    'alpha' : [0.0001,0.001,0.01,0.1],
    'learning_rate' : ['constant','adaptive'],
    'eta0' : [0.01,0.05,0.1,0.5],
    'max_iter' :[10000]

}

In [50]:
from sklearn.model_selection import GridSearchCV
gs1=GridSearchCV(sg,pm1,n_jobs=-1,cv=2,scoring='r2')
gs2=GridSearchCV(sg,pm2,n_jobs=-1,cv=2,scoring='r2')
gs1.fit(xtrain,ytrain)
gs2.fit(xtrain,ytrain)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SGDRegressor(...stopping=True)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'alpha': [0.0001, 0.001, ...], 'eta0': [0.01, 0.05, ...], 'l1_ratio': [0.1, 0.3, ...], 'learning_rate': ['constant', 'adaptive'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",T

In [ ]:
g1=r2_score(ytest,gs1.predict(xtest))
g2=r2_score(ytrain,gs1.predict(xtrain))

print(f"Test score {g1} , Train score {g2}")

g21=r2_score(ytest,gs2.predict(xtest))
g22=r2_score(ytrain,gs2.predict(xtrain))

print(f"Test score {g21} , Train score {g22}")

Test score 0.9877367400946107 , Train score 0.9889832519259542
Test score 0.9877229463475531 , Train score 0.9889803366024216


In [52]:
gs1.best_params_

{'alpha': 0.0001,
 'eta0': 0.05,
 'learning_rate': 'adaptive',
 'loss': 'squared_error',
 'max_iter': 10000,
 'penalty': 'l2'}

In [60]:
sgdt=SGDRegressor(alpha = 0.0001,
 eta0= 0.05,
 learning_rate = 'constant',
 loss = 'squared_error',
 max_iter= 10000,
  early_stopping=False,
 penalty= 'l2')

In [64]:
import random
X_tr = np.asarray(xtrain)
y_tr = np.asarray(ytrain).ravel()
X_te = np.asarray(xtest)
y_te = np.asarray(ytest).ravel()

batch = 25
epochs = 100 

for epoch in range(epochs):
    for i in range(X_tr.shape[0] // batch):
        ind = random.sample(range(X_tr.shape[0]), batch)
        sgdt.partial_fit(X_tr[ind], y_tr[ind])

g_1 = r2_score(y_te, sgdt.predict(X_te))
g_2 = r2_score(y_tr, sgdt.predict(X_tr))

print(f"Test score: {g_1}, Train score: {g_2}")



Test score: 0.9851589616440254, Train score: 0.9864483774718786
